# Plan #97 — structured attempt contract
Status: planned. Execution mode: fixture. This notebook freezes the cross-seam payload before production code.

In [ ]:
# Phase 1: provider response -> attempt write
# input: raw provider content + call context
# output: metadata-first attempt record
# acceptance: no raw body is stored; hash and typed failure survive
import hashlib
raw = '{"decision": {"action": "answer"}}'
attempt0 = {
    'logical_call_id': 'call_fixture_1', 'ordinal': 0,
    'trace_id': 'trace_fixture_1', 'task': 'planner',
    'model': 'provider/model', 'execution_path': 'native_schema',
    'schema_hash': 'schema_fixture',
    'raw_sha256': hashlib.sha256(raw.encode()).hexdigest(),
    'raw_artifact_ref': None, 'outcome': 'validation_failed',
    'failure_class': 'missing_required',
    'issues': [{'location': ['decision','rationale'], 'code': 'missing', 'message': 'Field required'}],
    'recovery_decision': 'retry',
}
assert 'raw_content' not in attempt0 and len(attempt0['raw_sha256']) == 64

In [ ]:
# Phase 2: retry -> ordered readback
# input: attempt 0 failure + attempt 1 success
# output: complete ordered attempt history and selected ordinal
# acceptance: attempt 0 cannot disappear when attempt 1 succeeds
attempt1 = {**attempt0, 'ordinal': 1, 'outcome': 'validated', 'failure_class': None, 'issues': [], 'recovery_decision': 'none'}
history = sorted([attempt1, attempt0], key=lambda item: item['ordinal'])
assert [item['outcome'] for item in history] == ['validation_failed', 'validated']
assert len(history) == 2 and history[-1]['ordinal'] == 1